# PCN Detection & Masking Pipeline

## Purpose
Scan entire tables (TB-scale) for Payment Card Numbers (PCNs), validate each candidate with the **Luhn algorithm**, and mask confirmed PCNs — all executed **distributedly on Spark workers** with zero driver-side data collection.

## Pipeline Flow
```
Source Table
  → Cast every column to STRING
  → Dynamic strategy selection based on column count:
      ≤100 cols: single-pass concat all columns with '||'
      >100 cols: chunked processing (groups of CHUNK_SIZE columns)
  → Pandas UDF per chunk/pass:
      1. Vectorized regex pre-filter (series.str.contains) — skips clean rows
      2. Row-level Luhn validation + masking (only on candidate rows)
  → Split masked string(s) back by '||'
  → Cast each column back to its original data type
  → Write to Delta target table
```

## Architecture Decisions

| Concern | Decision | Why |
|---|---|---|
| Driver memory | No `.collect()`, `.toPandas()`, or broadcast of row data | Source tables can be TBs; driver has ~16 GB RAM |
| UDF performance | **Pandas UDF** (vectorized) over standard Python UDF | Standard UDFs serialize one row at a time across JVM↔Python; Pandas UDFs batch via Apache Arrow (zero-copy columnar), giving 3–100× speedup on string-heavy workloads |
| UDF pre-filter | Vectorized `series.str.contains()` before row-level masking | Skips the expensive Luhn + regex-replace loop for rows with no PCN candidates (~95%+ of rows in typical data) |
| Wide tables | Chunked column processing when columns > `WIDE_TABLE_THRESHOLD` | Concatenating 100+ columns into one string creates massive strings that blow up regex scan time and per-batch memory; chunking bounds max string length |
| Execution model | All transformations are lazy until the final `.write()` | Lets Spark's Catalyst optimizer fuse, reorder, and prune stages across the full DAG |
| Partition skew | Full AQE configuration with skew auto-splitting | TB-scale tables often have skewed partitions; AQE detects and splits them at runtime |
| Output format | Delta Lake | ACID writes, schema enforcement, auto-compaction, Z-ORDER support |

---
## Setup, Imports & Spark Configuration

In [0]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, DataType, MapType, StringType, StructType
)
from typing import List, Dict
import pandas as pd
import re

SOURCE_TABLE = "capstone_project.pii_data.credit_card_transactions"
TARGET_TABLE = "capstone_project.pii_data.credit_card_transactions_masked"

# The delimiter used to join/split columns. "||" is chosen because it is
# extremely unlikely to appear in natural data (unlike "," or "|").
CONCAT_DELIMITER = "||"

# ---------------------------------------------------------------------------
# Wide-Table Chunking Parameters
# ---------------------------------------------------------------------------
# Tables with more columns than WIDE_TABLE_THRESHOLD are processed in chunks
# instead of a single concat-all pass. This avoids creating excessively long
# concatenated strings that blow up regex scan time and per-batch memory
# in the Pandas UDF.
WIDE_TABLE_THRESHOLD: int = 100

# Number of columns per chunk when chunked processing is active.
# Trade-off:
#   - LARGER chunks  → fewer UDF invocations (less Arrow transport overhead),
#                      but longer concatenated strings (more regex work)
#   - SMALLER chunks → shorter strings (faster regex), but more UDF
#                      invocations and more intermediate columns in the DAG
# 50 is a good balance: keeps concatenated strings under ~5 KB per row
# for typical column widths, while limiting chunk count to ~2–4 for most
# wide tables.
CHUNK_SIZE: int = 50

print(f"  Cell 1 complete — Spark configured for Arrow-backed Pandas UDFs.")
print(f"  Source: {SOURCE_TABLE}")
print(f"  Target: {TARGET_TABLE}")
print(f"  Wide-table threshold: {WIDE_TABLE_THRESHOLD} columns")
print(f"  Chunk size: {CHUNK_SIZE} columns")

---
## Pandas UDF: Regex + Luhn + Masking

### Why a Pandas UDF instead of a standard UDF?

| | Standard Python UDF | Pandas UDF (Vectorized) |
|---|---|---|
| **Granularity** | Called once **per row** | Called once **per batch** (default 10k rows) |
| **Serialization** | Pickle each scalar value across JVM→Python | Arrow zero-copy columnar transfer |
| **Python overhead** | Function-call overhead × N rows | Function-call overhead × (N / batch_size) |
| **Typical speedup** | baseline | **3–100×** for string/regex workloads |

For TB-scale data where every row requires regex scanning + Luhn math, this difference is the gap between a 2-hour job and a 20-minute job.

### Detection Methodology

1. **Regex scan** — find all 13–19 digit sequences (with optional space/hyphen separators)
2. **Luhn validation** — each candidate is stripped to pure digits and checked via the mod-10 checksum
3. **Masking** — only Luhn-valid matches get every character in their span replaced with `X`

### Vectorized Pre-Filter Optimization

Inside the Pandas UDF, we use a two-phase approach to avoid running expensive Python logic on every row:

1. **Phase 1 — Vectorized boolean mask:** `series.str.contains(pattern)` runs the regex across the entire Arrow batch using pandas' internal C/Cython string routines. This identifies which rows have ANY PCN candidates. For typical datasets, **95%+ of rows contain no card numbers** and are skipped entirely.
2. **Phase 2 — Row-level Luhn + masking:** Only the subset of rows with regex matches goes through the Python-level `_mask_pcns_in_string()` function for Luhn validation and character-level masking.

This hybrid approach gives us the best of both worlds: vectorized C-speed filtering for the common case, with precise Python logic only where needed.

In [0]:
# ============================================================================
# CELL 2: Pandas UDF — PCN Detection (Regex → Luhn) & Masking
# ============================================================================
#
# The detection logic (regex, Luhn, masking) lives in pcn_utils.py and the
# Pandas UDF wrapper in pcn_spark.py. This pipeline, the evaluation harness
# and method_comparison_benchmark.ipynb all import the same code, so
# evaluated and benchmarked behaviour is exactly what runs here.
#
# The UDF runs entirely on Spark worker nodes — the driver never sees row data.
# ============================================================================

from pcn_spark import make_mask_udf
from pcn_utils import mask_pcns_in_string

mask_pcns = make_mask_udf(mask_pcns_in_string)


print("✔ Cell 2 complete — Pandas UDF `mask_pcns` registered.")
print("  Detection: regex (13–19 digit sequences with optional separators)")
print("  Validation: Luhn mod-10 checksum")
print("  Masking: full character replacement with 'X'")
print("  Optimization: vectorized pre-filter skips clean rows")

---
## Distributed DataFrame Transformations

### Transformation DAG

#### Strategy A: Single-Pass (≤ `WIDE_TABLE_THRESHOLD` columns)

```
spark.table(SOURCE)                         ← lazy read (no data moves)
  │
  ├─ Snapshot schema (metadata only)        ← col names + types on driver (~KB)
  │
  ├─ Cast all columns to StringType         ← Spark-native, JVM-side
  ├─ concat_ws('||', col1, col2, ...)       ← Spark-native, JVM-side
  │
  ├─ mask_pcns(_concatenated) → _masked     ← Pandas UDF (Arrow batches to Python workers)
  │
  ├─ split(_masked, '||')                   ← Spark-native, JVM-side
  ├─ getItem(i) for each column             ← Spark-native, JVM-side
  ├─ Cast back to original types            ← Spark-native, JVM-side
  │
  └─ result_df (lazy — nothing executed yet)
```

#### Strategy B: Chunked Processing (> `WIDE_TABLE_THRESHOLD` columns)

```
spark.table(SOURCE)                         ← lazy read
  │
  ├─ Snapshot schema (metadata only)
  │
  ├─ For each chunk of CHUNK_SIZE columns:
  │   ├─ Cast chunk columns to StringType    ← JVM-side
  │   ├─ concat_ws('||', chunk_cols...)      ← JVM-side
  │   ├─ mask_pcns(chunk_concat)             ← Pandas UDF
  │   └─ (intermediate columns added to DAG)
  │
  ├─ Split each chunk's masked col & cast back
  │
  └─ result_df = select(all reconstructed columns)
```

The chunked strategy bounds the maximum concatenated string length regardless
of table width, preventing regex scan time and per-batch memory from growing
linearly with column count.

### NULL Handling
NULLs are replaced with the sentinel `__NULL__` before concatenation (because `concat_ws` skips NULLs, which would shift column positions). After splitting, sentinels are converted back to proper NULLs.

In [0]:
# ============================================================================
# CELL 3: Distributed PySpark DataFrame Transformations
# ============================================================================
#
# Every operation here is either:
#   (a) Spark-native (runs on JVM, zero Python overhead), or
#   (b) The Pandas UDF (Arrow-batched, runs on Python workers)
#
# No data is collected to the driver. The entire cell builds a lazy DAG;
# execution is deferred until the .write() call in Cell 4.
# ============================================================================


# ---------------------------------------------------------------------------
# Step 1: Read Source Table (Lazy)
# ---------------------------------------------------------------------------
# spark.table() registers a logical scan node in the DAG. No data is read
# from storage until an action (like .write() or .show()) triggers execution.
# For Delta tables, this also leverages data skipping and partition pruning
# if downstream filters exist.
source_df: DataFrame = spark.table(SOURCE_TABLE)


# ---------------------------------------------------------------------------
# Step 2: Snapshot the Original Schema (Metadata Only)
# ---------------------------------------------------------------------------
# .schema is a StructType object cached on the driver from the table's
# metadata (Hive metastore / Unity Catalog). This is a few KB even for
# tables with 1,000+ columns — no row data is transferred.
#
# We need this to:
#   1. Reconstruct the correct column order after splitting
#   2. Cast each column back to its original data type
original_schema: StructType = source_df.schema
col_names: List[str] = [field.name for field in original_schema.fields]
col_types: Dict[str, DataType] = {field.name: field.dataType for field in original_schema.fields}

num_cols: int = len(col_names)
print(f"Source schema: {num_cols} columns")
print(f"  Columns: {col_names}")
print(f"  Types: {[(f.name, str(f.dataType)) for f in original_schema.fields]}")


# ---------------------------------------------------------------------------
# Step 3: Dynamic Strategy Selection
# ---------------------------------------------------------------------------
# For narrow tables (≤ WIDE_TABLE_THRESHOLD columns), we use the original
# single-pass approach: concat all columns → UDF → split back.
#
# For wide tables (> WIDE_TABLE_THRESHOLD columns), concatenating all columns
# produces very long strings that:
#   - Increase regex scan time (O(n) in string length per pattern match)
#   - Increase per-batch memory in the Pandas UDF
#   - Risk exceeding Arrow's default string buffer limits
# Instead, we process chunks of CHUNK_SIZE columns independently.

# Sentinel for NULL preservation during concatenation (used in both strategies)
NULL_SENTINEL: str = "__NULL__"

# Regex-escaped delimiter for split operations
SPLIT_PATTERN: str = re.escape(CONCAT_DELIMITER)

# Spark cannot cast a split string directly back to ARRAY/MAP/STRUCT.
# Serialize complex values as JSON before masking and parse that JSON after
# splitting; scalar values retain the original string-cast round trip.
COMPLEX_TYPES = (ArrayType, MapType, StructType)


def serialize_for_masking(name: str):
    data_type = col_types[name]
    value = (
        F.to_json(F.col(name))
        if isinstance(data_type, COMPLEX_TYPES)
        else F.col(name).cast(StringType())
    )
    return F.coalesce(value, F.lit(NULL_SENTINEL))


def deserialize_after_masking(element, data_type: DataType):
    value = F.when(
        element == NULL_SENTINEL, F.lit(None).cast(StringType())
    ).otherwise(element)
    return (
        F.from_json(value, data_type)
        if isinstance(data_type, COMPLEX_TYPES)
        else value.cast(data_type)
    )


if num_cols <= WIDE_TABLE_THRESHOLD:
    # ==================================================================
    # Strategy A: Single-Pass (narrow tables)
    # ==================================================================
    print(f"\n→ Strategy A selected: single-pass concat ({num_cols} cols ≤ {WIDE_TABLE_THRESHOLD} threshold)")

    # --- Step 3A: Cast to String & Concatenate ---
    # Why COALESCE with a sentinel instead of letting NULLs through?
    #   concat_ws SKIPS null values. If col2 is NULL in a 3-column table:
    #     concat_ws('||', 'a', NULL, 'c') → 'a||c'  (only 1 delimiter!)
    #   When we later split by '||' expecting 3 elements, we'd get misaligned
    #   columns. The '__NULL__' sentinel preserves positional integrity:
    #     concat_ws('||', 'a', '__NULL__', 'c') → 'a||__NULL__||c'
    #
    # Why cast to StringType?
    #   concat_ws requires all inputs to be strings. Numeric, date, boolean,
    #   and other types must be cast first. Spark's .cast(StringType()) uses
    #   deterministic, locale-independent formatting, so the round-trip
    #   cast(string) → cast(original_type) is lossless for standard types.

    string_cols = [serialize_for_masking(c) for c in col_names]

    # concat_ws is a Spark-native expression — it runs on JVM workers as part of
    # the physical plan. No Python serialization involved.
    concat_df: DataFrame = source_df.withColumn(
        "_concatenated",
        F.concat_ws(CONCAT_DELIMITER, *string_cols)
    )

    # --- Step 4A: Apply the Pandas UDF ---
    # This is the ONLY step that crosses the JVM → Python boundary.
    # Spark serializes batches of the '_concatenated' column via Arrow,
    # sends them to Python workers, and receives the masked results back.
    #
    # Each batch is processed independently on whichever executor/core
    # Spark assigns it to — full parallelism across the cluster.
    masked_df: DataFrame = concat_df.withColumn(
        "_masked",
        mask_pcns(F.col("_concatenated"))
    )

    # --- Step 5A: Split Back into Original Columns & Cast to Original Types ---
    # F.split() and .getItem() are Spark-native expressions (JVM-side).
    # No additional Python UDF is needed for this reconstruction step.
    #
    # We escape the delimiter for the split regex because '||' has special
    # meaning in regex (alternation). re.escape('||') → '\\|\\|'.

    split_col = F.split(F.col("_masked"), SPLIT_PATTERN)

    select_exprs = []
    for idx, name in enumerate(col_names):
        # Extract the i-th element from the split array
        element = split_col.getItem(idx)

        # Restore NULLs and reconstruct the original scalar or complex type.
        element = deserialize_after_masking(element, col_types[name])

        # .alias() restores the original column name
        select_exprs.append(element.alias(name))

    # Select only the reconstructed columns (drops the intermediate
    # _concatenated and _masked helper columns)
    result_df: DataFrame = masked_df.select(select_exprs)

else:
    # ==================================================================
    # Strategy B: Chunked Processing (wide tables)
    # ==================================================================
    # Split columns into chunks of CHUNK_SIZE and process each chunk
    # independently: concat → UDF → split. This bounds the max
    # concatenated string length regardless of table width.
    #
    # Each chunk adds 2 withColumn calls (concat + UDF) to the DAG.
    # For a 500-column table with CHUNK_SIZE=50, that's 10 chunks × 2 = 20
    # plan nodes — well within Catalyst's capacity. The "withColumn in a
    # loop" anti-pattern only matters at hundreds of iterations.

    num_chunks: int = (num_cols + CHUNK_SIZE - 1) // CHUNK_SIZE
    print(f"\n→ Strategy B selected: chunked processing ({num_cols} cols > {WIDE_TABLE_THRESHOLD} threshold)")
    print(f"  Chunk size: {CHUNK_SIZE} columns, {num_chunks} chunks total")

    # Build column chunks: e.g., for 120 cols with CHUNK_SIZE=50:
    #   chunk 0: cols[0:50], chunk 1: cols[50:100], chunk 2: cols[100:120]
    chunks: List[List[str]] = [
        col_names[i : i + CHUNK_SIZE]
        for i in range(0, num_cols, CHUNK_SIZE)
    ]

    working_df: DataFrame = source_df
    select_exprs = []

    for chunk_idx, chunk_cols in enumerate(chunks):
        print(f"  Building DAG for chunk {chunk_idx}: {len(chunk_cols)} columns")

        # --- Concat this chunk's columns ---
        # Same NULL-sentinel + cast logic as Strategy A, but applied to
        # only CHUNK_SIZE columns at a time.
        chunk_string_cols = [serialize_for_masking(c) for c in chunk_cols]

        chunk_concat_col: str = f"_chunk_concat_{chunk_idx}"
        chunk_masked_col: str = f"_chunk_masked_{chunk_idx}"

        working_df = working_df.withColumn(
            chunk_concat_col,
            F.concat_ws(CONCAT_DELIMITER, *chunk_string_cols)
        )

        # --- Apply UDF to this chunk ---
        # The UDF processes a shorter string (max CHUNK_SIZE column values)
        # instead of the full row, reducing regex scan time proportionally.
        working_df = working_df.withColumn(
            chunk_masked_col,
            mask_pcns(F.col(chunk_concat_col))
        )

        # --- Build select expressions for this chunk's columns ---
        # Split the masked chunk string and extract each column by position.
        chunk_split = F.split(F.col(chunk_masked_col), SPLIT_PATTERN)

        for local_idx, name in enumerate(chunk_cols):
            element = chunk_split.getItem(local_idx)
            element = deserialize_after_masking(element, col_types[name])
            select_exprs.append(element.alias(name))

    # Final select: pick only the reconstructed original columns.
    # This drops all intermediate _chunk_concat_* and _chunk_masked_* columns.
    # Catalyst will also prune any upstream columns that aren't needed
    # to compute the selected expressions.
    result_df: DataFrame = working_df.select(select_exprs)


# ---------------------------------------------------------------------------
# Validation: Confirm schema matches source (both strategies)
# ---------------------------------------------------------------------------
assert result_df.columns == col_names, (
    f"Column order mismatch! "
    f"Expected {col_names}, got {result_df.columns}"
)

print("\n✔ Cell 3 complete — transformation DAG built (nothing executed yet).")
print("\nResult schema:")
result_df.printSchema()

---
## Optimized Delta Write

### Why Delta Lake?

| Feature | Benefit |
|---|---|
| **ACID transactions** | The write is atomic — readers never see partial/corrupt results |
| **Schema enforcement** | Delta validates the output matches the declared schema at write time |
| **Optimize Write** | Auto-coalesces many small output files into fewer, optimally-sized files (~128 MB). Without this, a high-parallelism Spark job can produce thousands of tiny files that degrade downstream read performance |
| **Auto Compaction** | Post-write background process that merges small files. Complements Optimize Write for incremental/streaming workloads |

### Execution
The `.saveAsTable()` call triggers Spark to execute the **entire lazy DAG** built in Cells 1–3. This is the only action in the notebook.

In [0]:
# ============================================================================
# CELL 4: Optimized Delta Write
# ============================================================================
#
# This is the ACTION that triggers execution of the full DAG.
# All the lazy transformations from Cells 1–3 are compiled by Spark's
# Catalyst optimizer into an optimized physical plan, then executed
# distributed across the cluster.
# ============================================================================

(
    result_df
    .write
    .format("delta")

    # -- Write Mode ---------------------------------------------------------
    # "overwrite" replaces the entire target table atomically.
    # For incremental pipelines, change to "append".
    .mode("overwrite")

    # -- Schema Evolution ---------------------------------------------------
    # If the source schema changes between runs (new columns, type changes),
    # this option lets the target table's schema evolve to match.
    # Remove this if you want strict schema enforcement (write fails on
    # schema mismatch).
    .option("overwriteSchema", "true")

    # -- Databricks-Specific Optimizations ----------------------------------
    # These options are silently ignored on open-source Delta Lake / Spark,
    # so the notebook remains portable.

    # Optimize Write: bin-packs small output files into ~128 MB target files.
    # Without this, a 1000-partition Spark job would write 1000 tiny files,
    # killing downstream read performance. With it, Spark adds an extra
    # shuffle stage to coalesce partitions before writing.
    .option("delta.autoOptimize.optimizeWrite", "true")

    # Auto Compaction: after the write completes, a background process
    # further merges any remaining small files. This is especially useful
    # for streaming/incremental workloads where Optimize Write alone can't
    # perfectly bin-pack due to arriving data volume.
    .option("delta.autoOptimize.autoCompact", "true")

    .saveAsTable(TARGET_TABLE)
)

print(f"✔ Cell 4 complete — masked data written to `{TARGET_TABLE}`.")
print(f"  Format: Delta")
print(f"  Mode: overwrite")

---
## Cell 5 — Validation (Optional)

Quick sanity check on a small sample. `.limit(5)` pushes a `LIMIT` into the scan — only 5 rows are materialized from storage.

In [0]:
# ============================================================================
# CELL 5 (Optional): Quick Validation
# ============================================================================
# Spot-check the masked output. .limit(5) ensures only 5 rows are read
# from storage — safe even for TB-scale tables.

print("Sample from masked target table:")
display(spark.table(TARGET_TABLE).limit(5))

# -- Row count comparison (optional) ----------------------------------------
# Uncomment to verify no rows were lost/duplicated in the pipeline.
# These are aggregate queries that scan metadata (Delta stats) when possible.
#
# source_count = spark.table(SOURCE_TABLE).count()
# target_count = spark.table(TARGET_TABLE).count()
# print(f"Source rows: {source_count:,}")
# print(f"Target rows: {target_count:,}")
# assert source_count == target_count, "Row count mismatch!"